In [ ]:
import IPython.display
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.visualization
import named_arrays as na
import ctis

In [ ]:
velocity = na.linspace(-300, 300, axis="wavelength", num=21) * u.km / u.s

In [ ]:
wavelength_rest = 171 * u.AA

In [ ]:
AA = dict(unit=u.AA, equivalencies=u.doppler_optical(wavelength_rest))

In [ ]:
wavelength = velocity.to(**AA)

In [ ]:
position_scene = na.Cartesian2dVectorLinearSpace(
    start=-10 * u.arcsec,
    stop=10 * u.arcsec,
    axis=na.Cartesian2dVectorArray("scene_x", "scene_y"),
    num=na.Cartesian2dVectorArray(64, 64),
)

In [ ]:
position_sensor = na.Cartesian2dVectorArray(
    x=na.arange(0, 64, axis="sensor_x") * u.pix,
    y=na.arange(0, 64, axis="sensor_y") * u.pix,
)

In [ ]:
coordinates_scene = na.SpectralPositionalVectorArray(wavelength, position_scene)
coordinates_sensor = na.SpectralPositionalVectorArray(wavelength, position_sensor)

In [ ]:
instrument = ctis.instruments.IdealInstrument(
    area_effective=(0.1 * u.mm**2).to(u.cm**2),
    timedelta_exposure=0.001 * u.s,
    plate_scale=.4 * u.arcsec / u.pix,
    dispersion=((10 * u.km / u.s).to(**AA) - wavelength_rest) / u.pix,
    angle=na.linspace(0, 360, num=36, axis="channel", endpoint=False) * u.deg,
    wavelength_ref=wavelength_rest,
    position_ref=32 * u.pix,
    coordinates_scene=coordinates_scene,
    coordinates_sensor=coordinates_sensor,
    axis_wavelength="wavelength",
    axis_scene_xy=("scene_x", "scene_y"),
    axis_sensor_xy=("sensor_x", "sensor_y"),
)

In [ ]:
center = na.SpectralPositionalVectorArray(
    wavelength=na.ScalarArray([-100, 0, 100] * u.km / u.s, axes="g").to(**AA),
    position=na.Cartesian2dVectorArray(
        x=0 * u.arcsec,
        y=na.ScalarArray([-7, 0, 7] * u.arcsec, axes="g"),
    )
)
width = na.SpectralPositionalVectorArray(
    wavelength=na.ScalarArray([50, 100, 75] * u.km / u.s, axes="g").to(**AA) - wavelength_rest,
    position=na.ScalarArray([1, 1, 1] * u.arcsec, axes="g"),
)
amplitude = 1 / np.sqrt(2 * np.pi) / width
radiance = amplitude * np.exp(-np.square(((coordinates_scene - center) / width)) / 2)
radiance = radiance.wavelength * radiance.position.x * radiance.position.y
radiance = radiance * u.erg / u.s / u.cm**2 * 5
radiance = radiance.cell_centers(("wavelength", "scene_x", "scene_y"))
radiance = radiance.sum("g")

scene = na.FunctionArray(
    inputs=coordinates_scene,
    outputs=radiance,
)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[.9,.1]),
        constrained_layout=True,
    )
    ax, cax = axs
    colorbar = na.plt.rgbmesh(
        C=scene,
        axis_wavelength="wavelength",
        ax=ax,
        vmin=0,
        vmax=scene.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_xlabel(f"scene $x$ ({ax.get_xlabel()})")
    ax.set_ylabel(f"scene $y$ ({ax.get_ylabel()})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

In [ ]:
image = instrument.image(scene.outputs)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[.9,.1]),
        constrained_layout=True,
    )
    ax, cax = axs
    label = "dispersion angle = " + instrument.angle.to_string_array("%03d")
    ani, colorbar = na.plt.rgbmovie(
        label,
        image.inputs.wavelength,
        image.inputs.position.x,
        image.inputs.position.y,
        C=image.outputs,
        axis_time="channel",
        axis_wavelength="wavelength",
        ax=ax,
        vmin=0,
        vmax=image.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_xlabel(f"sensor $x$ ({image.inputs.position.x.unit})")
    ax.set_ylabel(f"sensor $y$ ({image.inputs.position.y.unit})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

result = ani.to_jshtml(fps=10)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result

In [ ]:
image_avg = image.mean("wavelength")

In [ ]:
backprojected = instrument.backproject(image_avg.outputs)

In [ ]:
with astropy.visualization.quantity_support():
    fig, axs = plt.subplots(
        ncols=2,
        gridspec_kw=dict(width_ratios=[.9,.1]),
        constrained_layout=True,
    )
    ax, cax = axs
    ani, colorbar = na.plt.rgbmovie(
        label,
        backprojected.inputs.wavelength,
        backprojected.inputs.position.x,
        backprojected.inputs.position.y,
        C=backprojected.outputs,
        axis_time="channel",
        axis_wavelength="wavelength",
        ax=ax,
        vmin=0,
        vmax=backprojected.outputs.max(),
    )
    na.plt.pcolormesh(
        C=colorbar,
        axis_rgb="wavelength",
        ax=cax,
    )
    ax.set_xlabel(f"detector $x$ ({image.inputs.position.x.unit})")
    ax.set_ylabel(f"detector $y$ ({image.inputs.position.y.unit})")
    cax.xaxis.set_ticks_position("top")
    cax.xaxis.set_label_position("top")
    cax.yaxis.tick_right()
    cax.yaxis.set_label_position("right")

result = ani.to_jshtml(fps=10)
result = IPython.display.HTML(result)

plt.close(ani._fig)

result